<a href="https://colab.research.google.com/github/Mromererg/vindr-spineXR-detection/blob/main/01_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VinDr-SpineXR Veri Seti Hazırlama

Bu notebook, VinDr-SpineXR veri setinin YOLO eğitimi için hazırlanmasını kapsamaktadır:
- Google Drive mount
- COCO JSON → YOLO TXT format dönüşümü
- Dataset klasör yapısı oluşturma (symlink)
- dataset.yaml oluşturma

**Veri Seti:** VinDr-SpineXR (10,466 görüntü, 7 lezyon sınıfı)

**Sınıflar:**
- Osteophytes
- Spondylolysthesis
- Disc space narrowing
- Other lesions
- Surgical implant
- Foraminal stenosis
- Vertebral collapse

## 1. Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Veri Seti Yapısını İncele

In [ ]:
import os

BASE = '/content/drive/MyDrive/dataset/abnormal'

# Split klasörlerini listele
for root, dirs, files in os.walk(f'{BASE}/splits/original'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import pandas as pd

# Annotation istatistiklerini incele
train_df = pd.read_csv(f'{BASE}/splits/original/train_annotations_train.csv')
val_df = pd.read_csv(f'{BASE}/splits/original/train_annotations_val.csv')

print('Train annotation sayısı:', len(train_df))
print('Val annotation sayısı:', len(val_df))

print('\nTrain sınıf dağılımı:')
print(train_df['lesion_type'].value_counts())

print('\nVal sınıf dağılımı:')
print(val_df['lesion_type'].value_counts())

## 3. COCO JSON → YOLO TXT Format Dönüşümü

YOLO formatı: `class_id cx cy w h` (normalize edilmiş, 0-1 arası)

In [ ]:
import json
import os

def coco_to_yolo(coco_json_path, output_label_dir):
    """
    COCO JSON formatındaki annotation'ları YOLO TXT formatına dönüştürür.

    COCO bbox formatı: [x_min, y_min, width, height]
    YOLO bbox formatı: [cx, cy, w, h] (normalize edilmiş 0-1)
    """
    with open(coco_json_path) as f:
        data = json.load(f)

    os.makedirs(output_label_dir, exist_ok=True)

    # image_id → image info
    img_info = {img['id']: img for img in data['images']}

    # image_id → annotations
    anns_by_img = {img['id']: [] for img in data['images']}
    for ann in data['annotations']:
        if ann['image_id'] in anns_by_img:
            anns_by_img[ann['image_id']].append(ann)

    for img_id, info in img_info.items():
        W, H = info['width'], info['height']
        lines = []
        for ann in anns_by_img[img_id]:
            x, y, w, h = ann['bbox']  # COCO: x_min, y_min, w, h
            # YOLO: cx, cy, w, h (normalize)
            cx = (x + w / 2) / W
            cy = (y + h / 2) / H
            nw = w / W
            nh = h / H
            cls = ann['category_id']  # 0-indexed
            lines.append(f'{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')

        label_path = os.path.join(output_label_dir, info['file_name'].replace('.png', '.txt'))
        with open(label_path, 'w') as f:
            f.write('\n'.join(lines))

    print(f'Tamamlandı: {len(img_info)} görüntü için label oluşturuldu → {output_label_dir}')


# Train ve val label'larını oluştur
TRAIN_JSON = f'{BASE}/splits/original/train_coco_train.json'
VAL_JSON   = f'{BASE}/splits/original/train_coco_val.json'
TRAIN_LABELS = f'{BASE}/yolo_labels/train'
VAL_LABELS   = f'{BASE}/yolo_labels/val'

coco_to_yolo(TRAIN_JSON, TRAIN_LABELS)
coco_to_yolo(VAL_JSON,   VAL_LABELS)

In [ ]:
# Label sayılarını doğrula
print('Train label sayısı:', len(os.listdir(TRAIN_LABELS)))
print('Val label sayısı:',   len(os.listdir(VAL_LABELS)))

# Örnek label dosyasına bak
sample = os.listdir(TRAIN_LABELS)[0]
print(f'\nÖrnek label ({sample}):')
with open(os.path.join(TRAIN_LABELS, sample)) as f:
    print(f.read())

## 4. YOLO Dataset Klasör Yapısı Oluşturma

YOLO şu yapıyı bekler:
```
dataset/
  images/
    train/
    val/
  labels/
    train/
    val/
```

Görüntüleri kopyalamak yerine symlink oluşturuyoruz (disk alanı tasarrufu).

In [ ]:
import json

DATASET_DIR = '/content/spineXR_yolo_dataset_guncel'

os.makedirs(f'{DATASET_DIR}/images/train', exist_ok=True)
os.makedirs(f'{DATASET_DIR}/images/val',   exist_ok=True)
os.makedirs(f'{DATASET_DIR}/labels/train', exist_ok=True)
os.makedirs(f'{DATASET_DIR}/labels/val',   exist_ok=True)

with open(TRAIN_JSON) as f:
    train_data = json.load(f)
with open(VAL_JSON) as f:
    val_data = json.load(f)

# Image symlinks
for img in train_data['images']:
    src = f"{BASE}/train_pngs/{img['file_name']}"
    dst = f"{DATASET_DIR}/images/train/{img['file_name']}"
    if not os.path.exists(dst):
        os.symlink(src, dst)

for img in val_data['images']:
    src = f"{BASE}/train_pngs/{img['file_name']}"
    dst = f"{DATASET_DIR}/images/val/{img['file_name']}"
    if not os.path.exists(dst):
        os.symlink(src, dst)

# Label symlinks
for f in os.listdir(TRAIN_LABELS):
    src = f'{TRAIN_LABELS}/{f}'
    dst = f'{DATASET_DIR}/labels/train/{f}'
    if not os.path.exists(dst):
        os.symlink(src, dst)

for f in os.listdir(VAL_LABELS):
    src = f'{VAL_LABELS}/{f}'
    dst = f'{DATASET_DIR}/labels/val/{f}'
    if not os.path.exists(dst):
        os.symlink(src, dst)

print('Train images:', len(os.listdir(f'{DATASET_DIR}/images/train')))
print('Val images:',   len(os.listdir(f'{DATASET_DIR}/images/val')))
print('Train labels:', len(os.listdir(f'{DATASET_DIR}/labels/train')))
print('Val labels:',   len(os.listdir(f'{DATASET_DIR}/labels/val')))

## 5. dataset.yaml Oluşturma

In [ ]:
yaml_content = f"""path: {DATASET_DIR}

train: images/train
val: images/val

nc: 7
names:
  - Osteophytes
  - Spondylolysthesis
  - Disc space narrowing
  - Other lesions
  - Surgical implant
  - Foraminal stenosis
  - Vertebral collapse
"""

yaml_path = f'{DATASET_DIR}/dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print('dataset.yaml oluşturuldu:')
print(open(yaml_path).read())